# 01. Experimental Training: Bigrams & Hard IDF Filtering

**Experiment Objective:**  
Test whether adding bigrams (`ngram_range=(1, 2)`) and aggressive document frequency filtering (`max_df=0.10`) improves class separability and fixes the "Neutral Bleed" issue without using full TF-IDF.

**Experimental Setup:**
* **Unigrams + Bigrams:** `ngram_range=(1, 2)` captures contextual negation (e.g., "not good") and phrases.
* **Hard IDF Filter:** `max_df=0.10` drops tokens appearing in over 10% of documents to remove corpus-wide entity noise.
* **Frequency Threshold:** `min_df=10` filters hyper-rare word combinations to control vocabulary explosion.

In [1]:
import sys
import os
import pickle
import pandas as pd
import numpy as np

# Ensure Python can locate custom modules inside src/
sys.path.append(os.path.abspath("../../src"))

from preprocessing import fit_transform_train
from naive_bayes import MultinomialNaiveBayes

# 1. Load preprocessed training dataset
train_data_path = "../../data/processed/train_cleaned.csv"
df_train = pd.read_csv(train_data_path)

# Drop any null strings
df_train = df_train.dropna(subset=['tweet_content'])

print(f"Loaded training samples: {len(df_train)}")
print("\nClass Distribution:")
print(df_train['sentiment'].value_counts())

Loaded training samples: 57297

Class Distribution:
sentiment
Negative    21171
Positive    19078
Neutral     17048
Name: count, dtype: int64


## 2. N-Gram Feature Extraction

We extract both unigrams and bigrams while applying an aggressive `max_df` upper bound. This removes high-frequency non-sentiment tokens (game titles, platforms) that previously skewed Neutral classifications.

In [2]:
X_train_text = df_train['tweet_content']
y_train = df_train['sentiment'].values

# Fit vectorizer with Bigrams and Hard IDF boundaries
vectorizer, X_train_sparse = fit_transform_train(
    X_train_text, 
    min_df=10,           # Increased from 5 to prevent rare bigram noise
    max_df=0.1,          # Reduced from 0.85 to strip generic entities
    ngram_range=(1, 2)    # Extract single words AND word pairs
)

n_samples, n_features = X_train_sparse.shape
print(f"Experimental Matrix Shape: {n_samples} rows x {n_features} features")
print(f"Total Non-Zero Elements: {X_train_sparse.nnz}")
print(f"Sparsity Ratio: {100 * (1 - X_train_sparse.nnz / (n_samples * n_features)):.2f}%")

Experimental Matrix Shape: 57297 rows x 20068 features
Total Non-Zero Elements: 1132156
Sparsity Ratio: 99.90%


## 3. Multinomial Naive Bayes Training

We fit the custom `MultinomialNaiveBayes` instance on the expanded N-gram feature matrix using standard Laplace smoothing ($\alpha = 1.0$).

In [3]:
# Instantiate and fit experimental model
model = MultinomialNaiveBayes(alpha=1.0)
model.fit(X_train_sparse, y_train)

print("Experimental model training complete.")
print(f"Classes learned: {model.classes_}")

Experimental model training complete.
Classes learned: ['Negative' 'Neutral' 'Positive']


## 4. Parameter Inspection

We analyze the highest log-likelihood features across classes to verify if contextual bigrams are successfully learned and driving predictions.

In [4]:
params = model.get_learned_parameters()
feature_names = vectorizer.get_feature_names_out()

print("--- Top 0 2Features (Unigrams/Bigrams) per Class ---")
for c in params["classes"]:
    log_likelihoods = params["log_likelihoods"][c]
    top_10_idx = np.argsort(log_likelihoods)[-20:][::-1]
    
    print(f"\nClass: [{c.upper()}]")
    for idx in top_10_idx:
        feature = feature_names[idx]
        score = log_likelihoods[idx]
        print(f"  {feature:<20} | Log-Likelihood: {score:.4f}")

--- Top 0 2Features (Unigrams/Bigrams) per Class ---

Class: [NEGATIVE]
  not                  | Log-Likelihood: -5.2165
  can                  | Log-Likelihood: -5.2730
  me                   | Log-Likelihood: -5.3214
  have                 | Log-Likelihood: -5.3520
  are                  | Log-Likelihood: -5.3926
  all                  | Log-Likelihood: -5.3972
  just                 | Log-Likelihood: -5.3981
  but                  | Log-Likelihood: -5.4397
  be                   | Log-Likelihood: -5.5608
  they                 | Log-Likelihood: -5.5733
  your                 | Log-Likelihood: -5.5881
  why                  | Log-Likelihood: -5.5909
  mention mention      | Log-Likelihood: -5.6670
  get                  | Log-Likelihood: -5.6748
  like                 | Log-Likelihood: -5.6956
  was                  | Log-Likelihood: -5.7218
  no                   | Log-Likelihood: -5.7416
  shit                 | Log-Likelihood: -5.7967
  when                 | Log-Likelihood: -5.81

## 5. Artifact Preservation

We save the experimental model and vectorizer with distinct file names (`exp_*.pkl`) to keep the baseline artifacts intact.

In [5]:
models_dir = "../../models/Experimentation (Bigrams + Hard IDF)"
os.makedirs(models_dir, exist_ok=True)

# 1. Save experimental model binary
exp_model_path = os.path.join(models_dir, "exp_naive_bayes_model.pkl")
model.save_model(exp_model_path)
print(f"Saved experimental model state to: {exp_model_path}")

# 2. Save experimental vectorizer binary
exp_vectorizer_path = os.path.join(models_dir, "exp_vectorizer.pkl")
with open(exp_vectorizer_path, "wb") as f:
    pickle.dump(vectorizer, f)
print(f"Saved experimental vectorizer to: {exp_vectorizer_path}")

Saved experimental model state to: ../../models/Experimentation (Bigrams + Hard IDF)\exp_naive_bayes_model.pkl
Saved experimental vectorizer to: ../../models/Experimentation (Bigrams + Hard IDF)\exp_vectorizer.pkl
